# 第11回 似た国を、くり返しでまとめる（解答例・教員用）


---


## 前回からの接続


前回は、392台の車を主成分分析（principal component analysis）の地図に置き、2本の軸に名前を付けて読みました。地図の上では、アメリカ車と、日本・ヨーロッパ車が、左右に分かれて見えました。

けれども、地域の列が無かったら、どこで区切ればよいでしょうか。今日は、答えの列を使わずに、データをいくつかのグループに分ける方法を考えます。


---


## 今日の分析目標


あなたは国際協力機関の担当者です。答えの列が無い142か国を、どうやって3つのグループに分けるか、分けたグループは何を表すかに答える。

2007年の142か国の平均寿命と1人あたりGDPを使います。k-meansという方法を、まず1回ぶん書き、次に止まるまでくり返すように書き、scikit-learnの`KMeans`と答え合わせをします。そのあと、グループに名前を付け、境目にいる国を探します。

もう1つのねらいは、止まる条件を書いて、`while`で「収束（convergence）するまで」くり返せるようになることです。最後の「目標に答えられたか」で振り返りましょう。


### 場面

あなたは、国際協力機関の担当者です。世界の国々に、保健や経済の支援をしています。142か国に1か国ずつ方針を立てるのは大変なので、状況の似た国をまとめて、グループごとに方針の土台を作りたいと考えています。

どの国がどのグループかを示す答えの列は、どこにもありません。グループは、データから見つけるしかありません。


### 手元のデータ

Gapminder財団が公開している、世界の国々の統計です（plotly/datasets経由。CC-BY）。1行が、ある国のある年で、1952年から2007年まで5年ごとに、平均寿命・人口・1人あたりGDP（ドル）が並びます。

今日は、いちばん新しい2007年の142か国を使い、平均寿命と1人あたりGDPの2列で分けます。


### 最後に出す答え

142か国を分けた3つのグループと、それぞれの名前、そしてどのグループとも言いにくい、境目にいる国です。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- k-meansの手順（いちばん近い中心に割り当てる → 中心を重心に動かす → 動かなくなるまでくり返す）を、式とコードで説明できる
- `while`と止まる条件で、収束するまでくり返すコードを書ける。回数の上限も付けられる
- 慣性（中心までの距離の二乗和）がくり返しのたびに下がることを確かめ、必ず止まる理由を説明できる
- 自分で書いたk-meansを`KMeans`と比べ、出発点で答えが変わることを確かめられる
- グループを要約して名前を付け、境目にいる国を見つけられる


---


## 0. 準備


### 最初に実行する：道具を呼び出す

- 役割：このノートで使う道具（ライブラリ）を呼び出す
- 入力：なし
- 出力：なし（終わるとセルの左に ✓ が付く）


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


### 図の日本語を表示できるようにする

- 役割：図の中の日本語が □ にならないようにし、図の大きさと字の大きさをそろえる
- 入力：なし
- 出力：「セットアップ完了」の表示

`try:`から先は、日本語表示の道具が動かなくなったときの保険で、中身は読まなくてかまいません。


In [ ]:
# 日本語が図の中で □ になるのを防ぐ
!pip install -q japanize-matplotlib
try:
    import japanize_matplotlib
# japanize が動かなくなったときの保険：同梱フォントを直接登録する
except Exception:
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = None
    if spec:
        ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None)
    if ttf:
        fm.fontManager.addfont(str(ttf))
        name = fm.FontProperties(fname=str(ttf)).get_name()
        plt.rcParams['font.family'] = name

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False  # マイナス記号の化けを防ぐ
print('セットアップ完了')


### データの置き場所

- 役割：データの置き場所を決める
- 入力：なし
- 出力：なし


In [ ]:
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は、② のデータ解析（グループに分ける計算）を中心に進み、最後に ③（グループに名前を付ける）へ進みます。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


### 今日の道筋

次の6つの小さな問いに答えていきます。

1. 142か国は、どう散らばっているか？（1節）
2. 1回ぶんの手順は、どう書くか？（2節）
3. いつ止めればよいか？（3節）
4. `KMeans`と同じ答えになるか？（4節）
5. 3つのグループは、どんな国か？（5節）
6. どの国が、境目にいるか？（6節）


---


## 1. 142か国は、どう散らばっているか？


準備ができました。グループに分ける前に、まず142か国が、平均寿命と1人あたりGDPの平面に、どう散らばっているかを見ます。


### お手本：2007年の国を取り出す

- 役割：データを読み、2007年の行だけを取り出し、列の名前を日本語にする
- 入力：データの場所
- 出力：表の大きさと、先頭の3か国

`rename`は、列の名前を付け替えます。`{'元の名前': '新しい名前'}`の辞書で指定します。


In [ ]:
gm = pd.read_csv(f'{DATA_DIR}/gapminder.csv')
d = gm[gm['year'] == 2007].reset_index(drop=True)
d = d.rename(columns={'country': '国', 'continent': '大陸',
                      'lifeExp': '平均寿命', 'gdpPercap': 'GDP'})
print(d.shape)
d[['国', '大陸', '平均寿命', 'GDP']].head(3).round(1)


142か国です。1行が1か国になりました。`GDP`は1人あたりGDP（ドル）です。大陸ごとの国の数も見ておきます。


In [ ]:
d['大陸'].value_counts()


アフリカが52か国でいちばん多く、アジア33、ヨーロッパ30、アメリカ大陸25、オセアニア2と続きます。


### お手本：散布図で見る

- 役割：142か国を、横に1人あたりGDP、縦に平均寿命をとって点で描く
- 入力：表`d`
- 出力：散布図


In [ ]:
plt.scatter(d['GDP'], d['平均寿命'], s=15)
plt.xlabel('1人あたりGDP（ドル）')
plt.ylabel('平均寿命（年）')
plt.show()


左の端に、GDPの小さな国がぎっしり詰まり、右へは少しの国がまばらに伸びています。GDPは、数百ドルの国から4万ドルを超える国まであり、右に長く裾を引いた分布です。このままでは、左の端の国どうしの違いが見えません。


### 変えてみる：GDPを対数にする

- 役割：GDPの常用対数（10を底とする対数）の列`対数GDP`を足し、もう一度描く
- 入力：表`d`
- 出力：散布図

`np.log10`は、10を底とする対数です。対数GDPが3なら1000ドル、4なら10000ドルです。何倍違うかが、同じ幅で表されます。


In [ ]:
print(np.log10([1000, 10000]))
print(10 ** 3, 10 ** 4)


In [ ]:
d['対数GDP'] = np.log10(d['GDP'])
plt.scatter(d['対数GDP'], d['平均寿命'], s=15)
plt.xlabel('対数GDP')
plt.ylabel('平均寿命（年）')
plt.show()


点が、左下から右上へ帯のように並びました。豊かな国ほど、平均寿命が長い傾向です。左下には寿命の短い国が、右上には寿命の長い国が、帯の中で濃く集まっているように見えます。


### 物差しをそろえる：標準化

グループに分けるときは、国と国の距離で「似ている」を測ります。ところが、平均寿命と対数GDPでは、数の大きさがまったく違います。標準偏差（standard deviation）を見てみます。


In [ ]:
d[['平均寿命', '対数GDP']].std().round(2)


平均寿命の標準偏差は12.07、対数GDPは0.59です。このまま距離を測ると、平均寿命の違いばかりが効き、対数GDPの違いはほとんど効きません。前回の主成分分析と同じく、標準化（各列から平均を引き、標準偏差で割る）してから使います。

- 役割：2列を標準化（standardization）し、配列`X`にする
- 入力：表`d`の2列
- 出力：`X`の大きさと、列ごとの平均と標準偏差

`sc.fit_transform`は、平均と標準偏差を覚えて（fit）、変換する（transform）を1行でします（第7回）。`sc`は、あとで使うので残しておきます。


In [ ]:
sc = StandardScaler()
X = sc.fit_transform(d[['平均寿命', '対数GDP']])
print(X.shape)
print(X.mean(axis=0).round(2))
print(X.std(axis=0).round(2))


`X`は142行2列です。どちらの列も平均0、標準偏差1になりました。ここからは、この`X`で、142か国を3つのグループに分けます。


---


## 2. 1回ぶんの手順は、どう書くか？


物差しがそろいました。では、142か国を、どうやって3つに分ければよいでしょうか。まず、手順を式で書き、1回ぶんをコードに写します。


### k-meansの考え方

k-means（ケイ・ミーンズ）は、グループの数$K$を先に決め、各グループに1つずつ中心を置いて分ける方法です。今日は$K=3$です。

1. 割り当て：各国を、いちばん近い中心のグループに入れる
2. 更新：各グループの中心を、そのグループの国の重心（平均）に動かす

この2つを、中心が動かなくなるまでくり返します。最初の中心（出発点）は、人が決めます。今日は、日本・インド・ナイジェリアの3か国を出発点にします。


### 式で書く

国$i$の標準化した値を$\mathbf x_i$、グループ$k$の中心を$\mathbf c_k$とします。国$i$と中心$k$の距離の二乗は

$$d_{ik} = \|\mathbf x_i - \mathbf c_k\|^2 = (x_{i1} - c_{k1})^2 + (x_{i2} - c_{k2})^2$$

です。割り当ては、$d_{ik}$がいちばん小さい$k$を選びます。更新は、グループ$k$に入った国の平均を、新しい$\mathbf c_k$にします。


### お手本：出発点を決める

- 役割：日本・インド・ナイジェリアの行番号を探し、その3か国の`X`を最初の中心`C`にする
- 入力：表`d`と`X`
- 出力：行番号と、3つの中心

`isin`で選ぶと、行番号は表の並び（国名のアルファベット順）になります。中心0がインド、中心1が日本、中心2がナイジェリアです。


In [ ]:
start = d.index[d['国'].isin(['Japan', 'India', 'Nigeria'])]
print(list(start))
C = X[start]
C.round(2)


### お手本：距離の表を作る

- 役割：142か国それぞれと、3つの中心との距離の二乗を、142行3列の表`D`にする
- 入力：`X`と中心`C`
- 出力：表の大きさと、先頭の3か国

`np.zeros((142, 3))`で0の並んだ表を作り、`for`で中心ごとに1列ずつ埋めます。`X - C[k]`は、142か国それぞれから中心`k`を引きます。`** 2`で二乗し、`sum(axis=1)`で横（2つの列）に足します（第9回）。


In [ ]:
D = np.zeros((len(X), 3))
for k in range(3):
    D[:, k] = ((X - C[k]) ** 2).sum(axis=1)
print(D.shape)
D[:3].round(2)


1行目のアフガニスタンは、中心2（ナイジェリア）までの距離の二乗がいちばん小さい国です。表を横に見て、いちばん小さい列を選べば、割り当てになります。


### お手本：いちばん近い中心を選ぶ

- 役割：各国を、距離の二乗がいちばん小さい中心のグループに割り当てる
- 入力：距離の表`D`
- 出力：先頭の5か国の割り当てと、グループごとの国の数

`D.argmin(axis=1)`は、行ごとに、いちばん小さい値がある列の番号を返します。`np.bincount`は、0・1・2がそれぞれ何個あるかを数えます。


In [ ]:
lab = D.argmin(axis=1)
print(lab[:5])
print(np.bincount(lab))


### お手本：中心を重心に動かす

- 役割：各グループに入った国の平均を、新しい中心`C_new`にする
- 入力：`X`と割り当て`lab`
- 出力：新しい中心

`X[lab == k]`は、グループ`k`に入った国の行だけを取り出します（真偽値で行を選ぶ。第1回）。`mean(axis=0)`で、列ごとの平均（重心）を出します。


In [ ]:
C_new = np.zeros((3, 2))
for k in range(3):
    C_new[k] = X[lab == k].mean(axis=0)
C_new.round(2)


中心が動きました。とくに中心1（日本）は、日本という1か国の位置から、日本と同じグループに入った国々の真ん中へ、大きく動いています。


In [ ]:
# @title 図：1回目の割り当てと、中心の動き
plt.scatter(X[:, 0], X[:, 1], c=lab, s=15, cmap='tab10', vmax=9)
for k in range(3):
    plt.annotate('', xy=C_new[k], xytext=C[k],
                 arrowprops=dict(arrowstyle='->', lw=2))
plt.scatter(C[:, 0], C[:, 1], marker='s', s=80, c='white',
            edgecolors='black')
plt.scatter(C_new[:, 0], C_new[:, 1], marker='s', s=80, c='black')
plt.xlabel('平均寿命（標準化）')
plt.ylabel('対数GDP（標準化）')
plt.show()


### TODO①：2回目をやってみる

`C = C_new`として中心を入れ替え、距離の表・割り当て・新しい中心を、もう一度計算してください。グループごとの国の数と、中心がどれだけ動いたか（`np.abs(C_new - C).max()`）を表示しましょう。


In [ ]:
# TODO: 中心を入れ替えて、2回目の割り当てと更新をする

# 解答例①：2回目をやってみる
C = C_new
D = np.zeros((len(X), 3))
for k in range(3):
    D[:, k] = ((X - C[k]) ** 2).sum(axis=1)
lab = D.argmin(axis=1)
C_new = np.zeros((3, 2))
for k in range(3):
    C_new[k] = X[lab == k].mean(axis=0)
print(np.bincount(lab))
print(np.abs(C_new - C).max().round(3))


2回目の割り当ては、グループ0が42か国、1が61か国、2が39か国です。中心のずれの最大は0.103で、1回目より小さくなりました。けれども、まだ0ではありません。中心が動いたので、3回目をすれば、割り当てがまた変わるかもしれません。何回くり返せばよいのでしょうか。


---


## 3. いつ止めればよいか？


1回ぶんの手順が書けました。けれども、何回くり返せばよいかは、前もって分かりません。そこで、回数を決めずに「中心が動かなくなるまで」くり返す書き方を使います。


### おさらい：for

`for`は、決まった回数（並びの要素の数）だけくり返します。


In [ ]:
for k in range(3):
    print(k)


### while：条件が成り立つあいだ、くり返す

`while 条件:`は、条件が`True`のあいだ、下の字下げしたかたまりをくり返します。条件が`False`になったら、くり返しを抜けます。次の例は、`n`が3より小さいあいだ、`n`を表示して1増やします。


In [ ]:
n = 0
while n < 3:
    print(n)
    n = n + 1


`for`と違い、くり返す回数を前もって書きません。止まる条件を書きます。かたまりの中で条件を変えないと、いつまでも止まりません。


### np.allclose：ほぼ同じか

中心が「動かなくなった」かは、新しい中心と古い中心が同じかで判断します。小数の計算には、ごくわずかな誤差が入るので、`==`ではなく、`np.allclose`で「ほぼ同じか」を確かめます。


In [ ]:
print(0.1 + 0.2 == 0.3)
print(np.allclose(0.1 + 0.2, 0.3))
print(np.allclose([1.0, 2.0], [1.0, 2.1]))


`0.1 + 0.2`は、コンピュータの中ではわずかに0.3からずれるので、`==`では`False`になります。`np.allclose`なら、ごくわずかな違いは同じとみなして`True`です。0.1も違えば`False`です。`not`を前に付けると、`True`と`False`が入れ替わります。「動いた」は`not np.allclose(C_new, C)`です。


### 慣性：どれだけまとまっているか

くり返しのたびに、分け方がよくなっているかを測る物差しを決めます。各国から、自分のグループの中心までの距離の二乗を、142か国ぶん足したものを、慣性（inertia）と呼びます。

$$J = \sum_{i=1}^{142} \min_k d_{ik}$$

$\min_k d_{ik}$は、国$i$にとっていちばん近い中心までの距離の二乗です。コードでは`D.min(axis=1).sum()`です。慣性が小さいほど、各国が中心の近くにまとまっています。


### お手本：動かなくなるまで、くり返す

- 役割：2節の手順を、中心が動かなくなるまで`while`でくり返す。1回ごとに、回数と慣性を表示する
- 入力：`X`と出発点`start`
- 出力：回数と慣性の表示、最後の中心

`moved = True`から始めるので、1回目は必ずかたまりに入ります。中で`moved`を「中心が動いたか」に書き換え、動かなくなったら`False`になって抜けます。


In [ ]:
C = X[start]
n = 0
moved = True
while moved:
    D = np.zeros((len(X), 3))
    for k in range(3):
        D[:, k] = ((X - C[k]) ** 2).sum(axis=1)
    lab = D.argmin(axis=1)
    C_new = np.zeros((3, 2))
    for k in range(3):
        C_new[k] = X[lab == k].mean(axis=0)
    moved = not np.allclose(C_new, C)
    C = C_new
    n = n + 1
    print(n, D.min(axis=1).sum().round(2))


5回で止まりました。慣性は、84.85、57.98、56.9、55.56、55.4と、回るたびに下がっています。


### なぜ、必ず止まるのか

慣性は、2つの手順のどちらでも、増えることがありません。

- 割り当て：各国を、いちばん近い中心に入れ直すので、各国の距離の二乗は、増えない
- 更新：グループの中で、距離の二乗の和をいちばん小さくする点は、重心（平均）なので、中心を重心（centroid）に動かしても、増えない

142か国を3つに分ける分け方は、数は多いですが有限です。慣性が下がり続けるかぎり、同じ分け方には戻れないので、いつかは割り当てが変わらなくなり、中心も動かなくなります。


### TODO②：回数の上限を付ける

止まる理由はあっても、計算の誤差などで、いつまでも止まらないことに備えておくと安心です。`while`の条件に`and n < 3`を足し、3回で打ち切られることを確かめてください。`and`は、2つの条件の両方が`True`のときだけ`True`になります。


In [ ]:
# TODO: while の条件に回数の上限を足し、3回で打ち切る

# 解答例②：回数の上限を付ける
C = X[start]
n = 0
moved = True
while moved and n < 3:
    D = np.zeros((len(X), 3))
    for k in range(3):
        D[:, k] = ((X - C[k]) ** 2).sum(axis=1)
    lab = D.argmin(axis=1)
    C_new = np.zeros((3, 2))
    for k in range(3):
        C_new[k] = X[lab == k].mean(axis=0)
    moved = not np.allclose(C_new, C)
    C = C_new
    n = n + 1
    print(n, D.min(axis=1).sum().round(2))
print(moved)


3回目で、慣性が56.9のところで打ち切られました。中心はまだ動いていたので、`moved`は`True`のままです。ふだんは、上限を100のように大きくしておき、上限で止まったら（`moved`が`True`のままなら）、収束していないと気づけるようにします。


---


## 4. KMeansと同じ答えになるか？


自分で書いたk-meansが、5回で止まりました。では、この答えは、scikit-learnの`KMeans`と同じになるでしょうか。また、別の出発点から始めたら、同じ答えになるでしょうか。どちらを確かめるにも、3節のくり返しを、もう一度書くことになります。


### お手本：関数にまとめる

2回目に同じことをしたくなったので、関数にまとめます。

- 役割：3節のくり返しを、関数`my_kmeans`にまとめる
- 入力：標準化した値`X`と、出発点の中心`C`
- 出力：割り当て`lab`、最後の中心`C`、慣性`J`

中身は、3節のセルとほぼ同じです。違うのは、表示の代わりに`return`で3つの値を返すことと、`3`を`len(C)`（中心の数）にしたことです。


In [ ]:
def my_kmeans(X, C):
    """中心が動かなくなるまで、割り当てと更新をくり返す"""
    moved = True
    while moved:
        D = np.zeros((len(X), len(C)))
        for k in range(len(C)):
            D[:, k] = ((X - C[k]) ** 2).sum(axis=1)
        lab = D.argmin(axis=1)
        C_new = np.zeros(C.shape)
        for k in range(len(C)):
            C_new[k] = X[lab == k].mean(axis=0)
        moved = not np.allclose(C_new, C)
        C = C_new
    J = D.min(axis=1).sum()
    return lab, C, J


### お手本：関数を呼ぶ

- 役割：日本・インド・ナイジェリアを出発点にして`my_kmeans`を呼ぶ
- 入力：`X`と出発点`X[start]`
- 出力：グループごとの国の数と、慣性

3つの返す値を、`lab_a, C_a, J_a`の3つの名前で受け取ります（前回の`pca, S`と同じ）。


In [ ]:
lab_a, C_a, J_a = my_kmeans(X, X[start])
print(np.bincount(lab_a), round(J_a, 2))


慣性は55.4で、3節と同じです。


### お手本：同じ出発点からKMeansを回す

- 役割：日本・インド・ナイジェリアを出発点にして`KMeans`を回し、自作の答えと比べる
- 入力：`X`と出発点`X[start]`
- 出力：割り当てがすべて同じか、くり返しの回数、慣性

`init=`で出発点を渡し、`n_init=1`で、その出発点から1回だけ回します。`n_iter_`はくり返しの回数、`inertia_`は慣性です。


In [ ]:
km1 = KMeans(n_clusters=3, init=X[start], n_init=1).fit(X)
print((km1.labels_ == lab_a).all())
print(km1.n_iter_, round(km1.inertia_, 2))


142か国の割り当てが、すべて一致しました（`True`）。くり返しも5回、慣性も55.4で同じです。`KMeans`の中でも、今日書いた2つの手順がくり返されています。


### TODO③：表の先頭の3か国から始める

表の先頭の3か国（アフガニスタン・アルバニア・アルジェリア）の`X[:3]`を出発点にして`my_kmeans`を呼び、グループごとの国の数と慣性を表示してください。


In [ ]:
# TODO: 表の先頭の3か国を出発点にして my_kmeans を呼ぶ

# 解答例③：表の先頭の3か国から始める
lab_b, C_b, J_b = my_kmeans(X, X[:3])
print(np.bincount(lab_b), round(J_b, 2))


先頭の3か国から始めると、慣性は54.67で、日本・インド・ナイジェリアから始めたとき（55.4）と違います。グループの国の数も違い、別の分け方で止まりました。

k-meansは、慣性が下がらなくなったところで止まります。けれども、そこが、ありうる分け方の中で慣性がいちばん小さい答えとは限りません。止まる先は、出発点で変わります。


### お手本：出発点をいくつも試す

`KMeans`は、`n_init=10`とすると、出発点を変えて10回回し、慣性がいちばん小さい答えを残します。`random_state=0`は、出発点の選び方に使う乱数（random number）のシードを固定します。

- 役割：出発点を10通り試す`KMeans`で、142か国を3つに分ける
- 入力：`X`
- 出力：グループごとの国の数と、慣性


In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
print(np.bincount(km.labels_), round(km.inertia_, 2))


慣性は53.75で、自分で試した2つの出発点（55.4と54.67）のどちらよりも小さくなりました。ここからは、この`km`の分け方を使います。


---


## 5. 3つのグループは、どんな国か？


142か国が3つに分かれました。けれども、グループの番号0・1・2には、意味がありません。出発点が変われば、番号も入れ替わります。グループの中身を要約して、名前を付けます。


### おさらい：groupbyとagg

`groupby`で区分ごとに分け、`agg`で列ごとに計算をまとめます。`新しい列名=('列', '計算')`の形で、いくつも並べられます。


In [ ]:
d.groupby('大陸').agg(国数=('国', 'size'))


### お手本：グループの列を足す

- 役割：`km`の割り当てを、表`d`の列`グループ`にする
- 入力：`km.labels_`
- 出力：グループごとの国の数


In [ ]:
d['グループ'] = km.labels_
d['グループ'].value_counts()


### TODO④：グループを要約する

グループごとの国の数、平均寿命の平均、GDPの中央値を表示してください。GDPは右に裾を引いていたので、中央値で要約します。


In [ ]:
# TODO: グループごとの国の数・平均寿命の平均・GDPの中央値

# 解答例④：グループを要約する
d.groupby('グループ').agg(国数=('国', 'size'),
                          平均寿命=('平均寿命', 'mean'),
                          GDP中央値=('GDP', 'median')).round(1)


3つのグループは、国の数が52・51・39です。平均寿命の平均は53.0年・72.2年・78.8年、GDPの中央値は1,209ドル・7,093ドル・30,470ドルです（番号の順は、環境によって入れ替わることがあります）。平均寿命とGDPが、そろって段階的に上がっています。


### お手本：名前を付ける

- 役割：平均寿命の短い順に、グループに名前を付ける
- 入力：表`d`と`グループ`の列
- 出力：名前ごとの国の数

`sort_values().index`で、平均寿命の平均が短い順のグループ番号を取り出し、辞書で名前に対応させます。`map`は、辞書で値を置き換えます（前回の地域の名前と同じ）。番号が入れ替わっても、名前は同じグループに付きます。


In [ ]:
life = d.groupby('グループ')['平均寿命'].mean()
order = life.sort_values().index
name = {order[0]: '低寿命・低所得', order[1]: '中位',
        order[2]: '長寿・高所得'}
d['名前'] = d['グループ'].map(name)
d['名前'].value_counts()


In [ ]:
# @title 図：3つのグループ
for g in ['低寿命・低所得', '中位', '長寿・高所得']:
    m = d['名前'] == g
    plt.scatter(d.loc[m, '対数GDP'], d.loc[m, '平均寿命'], s=15,
                label=g)
plt.xlabel('対数GDP')
plt.ylabel('平均寿命（年）')
plt.legend()
plt.show()


### お手本：グループと大陸

- 役割：名前と大陸の組み合わせごとに、国の数を数える
- 入力：`d['名前']`と`d['大陸']`
- 出力：3行5列の表

`pd.crosstab`は、2つの区分の組み合わせごとに数を数えた表（クロス集計表）を作ります（第6回）。


In [ ]:
pd.crosstab(d['名前'], d['大陸'])


低寿命・低所得の52か国のうち44か国がアフリカ、長寿・高所得の39か国のうち23か国がヨーロッパです。中位の51か国には、アメリカ大陸20、アジア16、アフリカ8、ヨーロッパ7と、いろいろな大陸の国が入っています。

グループは大陸とかなり重なりますが、同じではありません。大陸の列は使っていないので、大陸を知らずに、平均寿命と豊かさだけから、似た国を集めたことになります。


---


## 6. どの国が、境目にいるか？


3つのグループに名前が付きました。けれども、グループの中の国が、どれも同じように典型的とは限りません。気になる国の位置と、グループの境目にいる国を確かめます。


### お手本：気になる国

- 役割：5か国のグループと、平均寿命・GDPを表示する
- 入力：表`d`
- 出力：5か国の表


In [ ]:
pick = ['Japan', 'China', 'India', 'Botswana', 'Equatorial Guinea']
d.loc[d['国'].isin(pick), ['国', '名前', '平均寿命', 'GDP']].round(1)


ボツワナと赤道ギニアは、GDPが12569.9ドルと12154.1ドルで、中位の多くの国より豊かです。けれども、平均寿命は50.7年と51.6年で短く、低寿命・低所得のグループに入りました。GDPだけで分けていたら、見落とした国です。支援の方針も、所得より保健を中心に考える必要がありそうです。


### km.transform：各中心までの距離

`km.transform(X)`は、各国から3つの中心までの距離（二乗しない距離）を、142行3列で返します。

- 役割：各国について、いちばん近い中心と、2番目に近い中心までの距離の差を出す
- 入力：`km`と`X`
- 出力：先頭の3か国の距離

`np.sort(..., axis=1)`は、行ごとに小さい順に並べ替えます。並べ替えた1列目がいちばん近い中心まで、2列目が2番目に近い中心までの距離です。


In [ ]:
dist = np.sort(km.transform(X), axis=1)
d['差'] = dist[:, 1] - dist[:, 0]
dist[:3].round(2)


差が小さい国ほど、2つの中心からほぼ同じ距離にいる、境目の国です。


### TODO⑤：境目にいる国

`差`の小さい順に並べ、境目にいる5か国の、国名・名前・平均寿命・GDP・差を表示してください。


In [ ]:
# TODO: 差の小さい順に5か国を表示する

# 解答例⑤：境目にいる国
cols = ['国', '名前', '平均寿命', 'GDP', '差']
d.sort_values('差')[cols].head(5).round(3)


境目にいるのは、トリニダード・トバゴ（差0.03）、イラク（0.065）、北朝鮮（0.086）、アルゼンチン（0.091）、メキシコ（0.111）です。トリニダード・トバゴは、中位と長寿・高所得の境目にいます。わずかな違いで、グループが入れ替わってもおかしくない国です。

k-meansは、どの国も必ずどこか1つのグループに入れます。境目の国は、グループの方針をそのまま当てはめず、1か国ずつ確かめる必要があります。


### 答え

2007年の142か国を、平均寿命と対数GDPを標準化して、k-meansで3つのグループに分けました。

- 低寿命・低所得（52か国）：平均寿命の平均53.0年、GDPの中央値1,209ドル。44か国がアフリカ
- 中位（51か国）：72.2年、7,093ドル。いろいろな大陸の国
- 長寿・高所得（39か国）：78.8年、30,470ドル。23か国がヨーロッパ

k-meansは、いちばん近い中心への割り当てと、重心への更新を、中心が動かなくなるまでくり返します。慣性が下がり続けるので必ず止まりますが、止まる先は出発点で変わるので、出発点を何通りも試して慣性のいちばん小さい答えを残します。

ボツワナや赤道ギニアのように、GDPは高くても寿命の短い国や、トリニダード・トバゴのように境目にいる国は、グループの方針をそのまま当てはめず、1か国ずつ確かめます。


まだ言えないこと

- 3つでよいか：グループの数$K=3$は、人が決めた。2つや4つのほうがよいかは、今日は確かめていない
- 分け方の確かさ：出発点や国の入れ替えで、分け方がどれくらい変わるかは、まだ測っていない
- 2列だけで分けた：教育や医療の列を足せば、グループは変わるかもしれない


---


## 目標に答えられたか


- 今日の目標は「答えの列が無い142か国を、どうやって3つのグループに分けるか、分けたグループは何を表すか」でした。3つのグループの名前と、分け方の手順を答えられますか？
- 1節で、GDPを対数にし、2列を標準化したのはなぜですか？
- 2節で、割り当てと更新は、それぞれ何をしましたか？
- 3節で、`while`はいつ止まりましたか？ 必ず止まると言えるのはなぜですか？
- 4節で、出発点を変えると何が起きましたか？`n_init`は何をしますか？
- 5節で、グループの番号ではなく名前で読むのはなぜですか？
- 6節で、境目の国は、どうやって見つけましたか？


---


## 今日の要点


- 今日、持ち帰る1文：k-meansは「いちばん近い中心に割り当てる → 中心を重心に動かす」を、中心が動かなくなるまでくり返す。慣性は回るたびに下がるので必ず止まるが、止まった先は出発点で変わる
- 距離で分けるので、単位や数の大きさが違う列は、標準化してから使う。右に裾を引く列は、対数にしてから
- 慣性（中心までの距離の二乗和）は、割り当てでも更新でも増えない
- `KMeans`は、出発点を`n_init`通り試し、慣性のいちばん小さい答えを残す
- グループの番号に意味はない。要約して名前を付け、境目の国は1か国ずつ確かめる
- Python：`while 条件:`は、条件が`True`のあいだくり返す。止まる条件をかたまりの中で書き換え、回数の上限も付けておく。2回目に同じことをしたくなったら、`def`でまとめる


---


## 次回へ


今日は、自分でk-meansを書き、`KMeans`の中で何がくり返されているかを確かめました。中身を知っている道具なら、ほかの人が書いたコードの中に出てきても読めます。

次回の問いは、研究者が論文のために書いて公開したコードを、どこから、どう読めばよいか、です。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

10年前の1997年には、各国はどのグループにいたでしょうか。2007年に作った物差し（標準化の`sc`）と3つの中心（`km`）をそのまま使って、1997年の142か国を割り当て直してください。

`km.predict`は、各国をいちばん近い中心のグループに割り当てます（2節の「割り当て」の1段だけをします）。1997年の割り当てを名前にした列`d['1997年']`を作り、2007年の名前`d['名前']`とのクロス集計表と、グループが変わった国の数を表示しましょう。1997年の表も、2007年と同じ142か国が同じ順に並んでいます。


In [ ]:
d97 = gm[gm['year'] == 1997].reset_index(drop=True)
d97 = d97.rename(columns={'lifeExp': '平均寿命'})
d97['対数GDP'] = np.log10(d97['gdpPercap'])
X97 = sc.transform(d97[['平均寿命', '対数GDP']])
d['1997年'] = pd.Series(km.predict(X97)).map(name)
print(pd.crosstab(d['1997年'], d['名前']))
print((d['名前'] != d['1997年']).sum())


<details><summary>詰まったら</summary>

1節と同じように、`gm[gm['year'] == 1997]`で取り出し、`rename`で平均寿命の列名を付け、`対数GDP`の列を足します。
`X97 = sc.transform(d97[['平均寿命', '対数GDP']])`で、2007年の平均と標準偏差で標準化します（`fit`しない）。
`pd.Series(km.predict(X97)).map(name)`で名前にし、`(d['名前'] != d['1997年']).sum()`で数えます。

</details>


### 応用②（判断）

1997年から2007年のあいだに、より下のグループ（長寿・高所得 → 中位、中位 → 低寿命・低所得）へ移った国はどこですか。国名を英語で答えてください（例: Japan）。


In [ ]:
rank = {'低寿命・低所得': 0, '中位': 1, '長寿・高所得': 2}
down = d['名前'].map(rank) < d['1997年'].map(rank)
print(d.loc[down, ['国', '1997年', '名前', '平均寿命']])
print(d97.loc[down, ['country', '平均寿命']])
print('答え:', d.loc[down, '国'].values[0])


<details><summary>詰まったら</summary>

名前を順位に直す辞書`rank = {'低寿命・低所得': 0, '中位': 1, '長寿・高所得': 2}`を作り、`d['名前'].map(rank) < d['1997年'].map(rank)`で下に移った国を選びます。

</details>


### 応用③（解釈）

国際協力機関の担当者として、今日の3つのグループと、1997年からの移り変わりを、上司に3行程度で報告してください。グループの分け方、移った国から分かること、グループの方針をそのまま当てはめにくい国、を入れましょう。


（模範例）

2007年の142か国を、平均寿命と1人あたりGDPから、低寿命・低所得（52か国）、中位（51か国）、長寿・高所得（39か国）の3つに分けました。

同じ物差しで1997年に当てはめると、10年でグループが変わった国は9か国で、インドやポーランドなど8か国は上のグループへ移り、下へ移ったのは南アフリカだけでした。南アフリカは平均寿命が60.2年から49.3年に下がっており、所得より保健の支援を優先すべき国です。

ボツワナや赤道ギニアのように所得の割に寿命の短い国や、トリニダード・トバゴのように境目にいる国は、グループの方針をそのまま当てはめず、1か国ずつ確かめます。


---


## 発展（任意）


### HDBSCAN：Kを決めず、まとまらない国は外れ値にする

k-meansでは、人が決めることが2つありました。グループの数$K$を先に決めることと、すべての国をどこかのグループに必ず入れることです。境目の国も、まばらな場所にいる国も、どこかのグループに入れられます。

HDBSCAN（エイチ・ディー・ビー・スキャン）は、点が密に集まっている場所をグループとし、まばらな場所の点は外れ値（ラベル −1）として残す方法です。$K$は決めません。決めるのは、グループと認める最小の国の数（`min_cluster_size`）です。

- 役割：同じ`X`をHDBSCANで分け、グループの数と外れ値の数、グループごとの要約を出す
- 入力：`X`と表`d`
- 出力：グループの数、外れ値の数、要約の表


In [ ]:
from sklearn.cluster import HDBSCAN
hdb = HDBSCAN(min_cluster_size=8, copy=True).fit_predict(X)
d['HDB'] = hdb
print('グループの数:', hdb.max() + 1)
print('外れ値:', (hdb == -1).sum(), 'か国')
d.groupby('HDB').agg(国数=('国', 'size'),
                     平均寿命=('平均寿命', 'mean'),
                     GDP中央値=('GDP', 'median')).round(1)


In [ ]:
for g in sorted(d['HDB'].unique()):
    m = d['HDB'] == g
    if g == -1:
        plt.scatter(d.loc[m, '対数GDP'], d.loc[m, '平均寿命'], s=25,
                    marker='x', color='gray', label='外れ値')
    else:
        plt.scatter(d.loc[m, '対数GDP'], d.loc[m, '平均寿命'], s=15,
                    label=f'グループ{g}')
plt.xlabel('対数GDP')
plt.ylabel('平均寿命（年）')
plt.legend()
plt.show()


### 外れ値になった国

- 役割：外れ値になった国のうち、k-meansの名前ごとの数と、平均寿命の短い5か国を表示する
- 入力：表`d`
- 出力：名前ごとの数と、5か国の表


In [ ]:
out = d[d['HDB'] == -1]
print(out['名前'].value_counts())
out.sort_values('平均寿命')[['国', '名前', '平均寿命', 'GDP']].head(5)


外れ値には、2種類の国が混ざります。1つは、図の左下の、まばらな場所にいる国です。平均寿命のとても短い国々は、たがいに離れて散らばっているので、密なグループになりません。もう1つは、グループとグループのあいだにいる国です。

どこにも無理に入れないことで、1か国ずつ確かめるべき国が、外れ値として見えてきます。

試すなら、`min_cluster_size`を5や15に変えて、グループの数と外れ値の数がどう変わるかを見てみましょう。$K$を決めない代わりに、何を「密」とみなすかを決めていることが分かります。
